# Setup the Environment

In [ ]:
!nvidia-smi
!pip install -q transformers datasets accelerate scikit-learn

Wed Nov 18 04:28:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Connected to Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


# Load the Data

In [ ]:
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)

DATA = "/content/drive/MyDrive/fake-news/processed_v3"
OUT = "/content/drive/MyDrive/fake-news/models/distilroberta_v3"
MODEL_NAME = "distilroberta-base"
MAX_LEN = 256   # only the first 256 tokens of each article are used

print("GPU available:", torch.cuda.is_available())


def load(split):
    df = pd.read_csv(f"{DATA}/{split}.csv")
    df["title"] = df["title"].fillna("")
    df["text"] = df["text"].fillna("")
    df["full"] = df["title"] + ". " + df["text"]
    return df


# Label meaning: 0 = real, 1 = fake.
# The test set is deliberately NOT loaded here.
train_df, val_df = load("train"), load("val")
print(len(train_df), len(val_df))

GPU available: True
48918 6115


# Tokenize

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_dataset(df):
    ds = Dataset.from_dict({"text": df["full"].tolist(), "labels": df["label"].tolist()})
    return ds.map(
        lambda batch: tokenizer(batch["text"], truncation=True, max_length=MAX_LEN),
        batched=True,
        remove_columns=["text"],
    )

train_ds, val_ds = to_dataset(train_df), to_dataset(val_df)

config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Map:   0%|          | 0/48918 [00:00<?, ? examples/s]

Map:   0%|          | 0/6115 [00:00<?, ? examples/s]

# Train the Model

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: "real", 1: "fake"},
    label2id={"real": 0, "fake": 1},
)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
    }


# 48,918 articles / batch size 16 = about 3,057 steps per epoch, so 2 epochs = about 6,100 steps.
# warmup_steps=600 is about 10% of that: the learning rate rises slowly for the first 600 steps.
args = TrainingArguments(
    output_dir="/content/checkpoints",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=600,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    save_total_limit=1,
    fp16=True,
    logging_steps=200,
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)
trainer.train()

model.safetensors: reconstructing file:   0%|          |  0.00B /  331MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.020927,0.033734,0.992968,0.992847
2,0.004201,0.022982,0.995912,0.995845


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=6116, training_loss=0.045272178523988016, metrics={'train_runtime': 726.3093, 'train_samples_per_second': 134.703, 'train_steps_per_second': 8.421, 'total_flos': 6480040207454208.0, 'train_loss': 0.045272178523988016, 'epoch': 2.0})

# Save the Model

In [ ]:
pred = trainer.predict(val_ds)
y_pred = np.argmax(pred.predictions, axis=-1)
y_true = pred.label_ids

print(classification_report(y_true, y_pred, target_names=["real", "fake"], digits=4))
print("Confusion matrix (rows = true, columns = predicted; order: real, fake):")
print(confusion_matrix(y_true, y_pred))

trainer.save_model(OUT)
tokenizer.save_pretrained(OUT)
print("Saved to", OUT)

              precision    recall  f1-score   support

        real     0.9954    0.9974    0.9964      3441
        fake     0.9966    0.9940    0.9953      2674

    accuracy                         0.9959      6115
   macro avg     0.9960    0.9957    0.9958      6115
weighted avg     0.9959    0.9959    0.9959      6115

Confusion matrix (rows = true, columns = predicted; order: real, fake):
[[3432    9]
 [  16 2658]]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to /content/drive/MyDrive/fake-news/models/distilroberta_v3


# Load saved model from Drive

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(OUT)
trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir="/content/tmp",
        per_device_eval_batch_size=64,
        fp16=True,
        report_to="none",
    ),
    data_collator=DataCollatorWithPadding(tokenizer),
)
print("Model loaded from Drive")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Model loaded from Drive


# Robustness check: style and formatting stress test

Re-evaluates the same validation articles after removing style and formatting cues (case, punctuation, title, first 300 characters) to see how much the model relies on them.

In [ ]:
import re


def normalize(text):
    """Lowercase, plain quotes, no punctuation: removes style and formatting cues."""
    text = text.lower()
    text = text.replace("’", "'").replace("‘", "'").replace("“", '"').replace("”", '"')
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def run_variant(name, full_series):
    df = val_df.copy()
    df["full"] = full_series
    ds = to_dataset(df)
    out = trainer.predict(ds)
    preds = np.argmax(out.predictions, axis=-1)
    acc = accuracy_score(df["label"], preds)
    f1 = f1_score(df["label"], preds, average="macro")
    print(f"{name:38s} accuracy={acc:.4f}  macro_f1={f1:.4f}")


run_variant("1. original", val_df["full"])
run_variant("2. lowercase, no punctuation", val_df["full"].apply(normalize))
run_variant("3. title only", val_df["title"])
run_variant("4. body only (no title)", val_df["text"])
run_variant("5. body without first 300 chars", val_df["text"].str[300:])

Map:   0%|          | 0/6115 [00:00<?, ? examples/s]

1. original                            accuracy=0.9959  macro_f1=0.9958


Map:   0%|          | 0/6115 [00:00<?, ? examples/s]

2. lowercase, no punctuation           accuracy=0.7452  macro_f1=0.7419


Map:   0%|          | 0/6115 [00:00<?, ? examples/s]

3. title only                          accuracy=0.9066  macro_f1=0.9063


Map:   0%|          | 0/6115 [00:00<?, ? examples/s]

4. body only (no title)                accuracy=0.9797  macro_f1=0.9793


Map:   0%|          | 0/6115 [00:00<?, ? examples/s]

5. body without first 300 chars        accuracy=0.9195  macro_f1=0.9178


# Content-only experiment

In [ ]:
OUT_NORM = "/content/drive/MyDrive/fake-news/models/distilroberta_v3_normalized"

# Normalize the text for training AND validation (same function as the robustness check)
train_norm = train_df.copy()
val_norm = val_df.copy()
train_norm["full"] = train_norm["full"].apply(normalize)
val_norm["full"] = val_norm["full"].apply(normalize)
train_ds_n, val_ds_n = to_dataset(train_norm), to_dataset(val_norm)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
    }


model_n = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: "real", 1: "fake"},
    label2id={"real": 0, "fake": 1},
)

args_n = TrainingArguments(
    output_dir="/content/checkpoints_norm",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=600,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    save_total_limit=1,
    fp16=True,
    logging_steps=200,
    report_to="none",
    seed=42,
)

trainer_n = Trainer(
    model=model_n,
    args=args_n,
    train_dataset=train_ds_n,
    eval_dataset=val_ds_n,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)
trainer_n.train()

# Evaluate on the normalized validation set and save to Drive
pred = trainer_n.predict(val_ds_n)
y_pred = np.argmax(pred.predictions, axis=-1)
print(classification_report(pred.label_ids, y_pred, target_names=["real", "fake"], digits=4))
print(confusion_matrix(pred.label_ids, y_pred))

trainer_n.save_model(OUT_NORM)
tokenizer.save_pretrained(OUT_NORM)
print("Saved to", OUT_NORM)

Map:   0%|          | 0/48918 [00:00<?, ? examples/s]

Map:   0%|          | 0/6115 [00:00<?, ? examples/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  331MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


... (2 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

              precision    recall  f1-score   support

        real     0.9748    0.9762    0.9755      3441
        fake     0.9693    0.9675    0.9684      2674

    accuracy                         0.9724      6115
   macro avg     0.9720    0.9718    0.9719      6115
weighted avg     0.9724    0.9724    0.9724      6115

[[3359   82]
 [  87 2587]]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to /content/drive/MyDrive/fake-news/models/distilroberta_v3_normalized


# External evaluation on LIAR (statements never seen in training)


In [ ]:
import re

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             f1_score, roc_auc_score)
from transformers import AutoTokenizer, AutoModelForSequenceClassification

BASE = "/content/drive/MyDrive/fake-news"
df = pd.read_csv(f"{BASE}/liar/liar_eval.csv")
y = df["label"].values  # 0 = real, 1 = fake
statements = df["statement"].tolist()
device = "cuda" if torch.cuda.is_available() else "cpu"
print("statements:", len(df), "| device:", device)


def normalize(text):
    """Same function used to train the normalized model."""
    text = text.lower()
    text = text.replace("’", "'").replace("‘", "'").replace("“", '"').replace("”", '"')
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def predict(model_dir, texts, batch_size=64):
    """Return the probability that each text is fake."""
    tok = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir).to(device).eval()
    probs = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = tok(texts[i:i + batch_size], truncation=True, max_length=256,
                        padding=True, return_tensors="pt").to(device)
            logits = model(**batch).logits
            probs.append(torch.softmax(logits, dim=-1)[:, 1].cpu().numpy())
    return np.concatenate(probs)


runs = {
    "original model": (f"{BASE}/models/distilroberta_v3", statements),
    "normalized model": (f"{BASE}/models/distilroberta_v3_normalized",
                         [normalize(s) for s in statements]),
}

majority = int(np.bincount(y).argmax())
print(f"\nalways predicting '{['real', 'fake'][majority]}' would score: {(y == majority).mean():.4f}")

for name, (path, texts) in runs.items():
    prob_fake = predict(path, texts)
    pred = (prob_fake >= 0.5).astype(int)
    print(f"\n===== {name} =====")
    print(f"accuracy: {accuracy_score(y, pred):.4f} | macro F1: {f1_score(y, pred, average='macro'):.4f}"
          f" | ROC AUC: {roc_auc_score(y, prob_fake):.4f}")
    print(f"share labelled fake: {pred.mean():.2%} (truly fake: {y.mean():.2%})")
    print(classification_report(y, pred, target_names=["real", "fake"], digits=4))
    print("Confusion matrix (rows = true, columns = predicted; order: real, fake):")
    print(confusion_matrix(y, pred))

statements: 8072 | device: cuda

always predicting 'real' would score: 0.5606


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]


===== original model =====
accuracy: 0.5673 | macro F1: 0.4866 | ROC AUC: 0.5322
share labelled fake: 16.43% (truly fake: 43.94%)
              precision    recall  f1-score   support

        real     0.5765    0.8594    0.6901      4525
        fake     0.5204    0.1945    0.2832      3547

    accuracy                         0.5673      8072
   macro avg     0.5484    0.5270    0.4866      8072
weighted avg     0.5518    0.5673    0.5113      8072

Confusion matrix (rows = true, columns = predicted; order: real, fake):
[[3889  636]
 [2857  690]]


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]


===== normalized model =====
accuracy: 0.4642 | macro F1: 0.4015 | ROC AUC: 0.5473
share labelled fake: 88.43% (truly fake: 43.94%)
              precision    recall  f1-score   support

        real     0.6071    0.1253    0.2077      4525
        fake     0.4455    0.8965    0.5952      3547

    accuracy                         0.4642      8072
   macro avg     0.5263    0.5109    0.4015      8072
weighted avg     0.5361    0.4642    0.3780      8072

Confusion matrix (rows = true, columns = predicted; order: real, fake):
[[ 567 3958]
 [ 367 3180]]


# Confidence analysis of the normalized model (validation set only)

In [ ]:
import re

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

BASE = "/content/drive/MyDrive/fake-news"
MODEL_DIR = f"{BASE}/models/distilroberta_v3_normalized"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)


def normalize(text):
    """Same function used to train the normalized model."""
    text = text.lower()
    text = text.replace("’", "'").replace("‘", "'").replace("“", '"').replace("”", '"')
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


val = pd.read_csv(f"{BASE}/processed_v3/val.csv")
val["title"] = val["title"].fillna("")
val["text"] = val["text"].fillna("")
texts = [normalize(t + ". " + x) for t, x in zip(val["title"], val["text"])]
y = val["label"].values  # 0 = real, 1 = fake

tok = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(device).eval()

probs = []
with torch.no_grad():
    for i in range(0, len(texts), 64):
        batch = tok(texts[i:i + 64], truncation=True, max_length=256,
                    padding=True, return_tensors="pt").to(device)
        logits = model(**batch).logits
        probs.append(torch.softmax(logits, dim=-1)[:, 1].cpu().numpy())
p_fake = np.concatenate(probs)

pd.DataFrame({"label": y, "p_fake": p_fake}).to_csv(
    f"{BASE}/models/val_probs_normalized.csv", index=False)

pred = (p_fake >= 0.5).astype(int)
print(f"\nvalidation accuracy: {(pred == y).mean():.4f}")

# 1. Calibration: when the model says "p% fake", how often is the article really fake?
print("\n--- Calibration ---")
edges = np.linspace(0, 1, 11)
ece = 0.0
for lo, hi in zip(edges[:-1], edges[1:]):
    mask = (p_fake >= lo) & ((p_fake < hi) if hi < 1 else (p_fake <= hi))
    if mask.sum() == 0:
        continue
    actual, said = y[mask].mean(), p_fake[mask].mean()
    ece += mask.mean() * abs(actual - said)
    print(f"model said {lo:.1f}-{hi:.1f} ({mask.sum():4d} articles): "
          f"average claimed {said:.2f}, really fake {actual:.2f}")
print(f"average gap between claimed and real (ECE): {ece:.4f}   (0 = perfectly honest)")

# 2. Abstention: answer only when the model is confident enough
print("\n--- Abstention: accuracy on the articles we still answer ---")
conf = np.maximum(p_fake, 1 - p_fake)
for t in (0.5, 0.7, 0.8, 0.9, 0.95, 0.99):
    mask = conf >= t
    print(f"confidence >= {t:.2f}: answered {mask.mean():6.1%} | accuracy {(pred[mask] == y[mask]).mean():.4f}")

# 3. Does article length matter?
print("\n--- Accuracy by article length ---")
words = np.array([len(t.split()) for t in val["text"]])
for lo, hi in [(0, 60), (60, 120), (120, 250), (250, 500), (500, 10**6)]:
    mask = (words >= lo) & (words < hi)
    if mask.sum():
        print(f"{lo:4d}-{hi if hi < 10**6 else 'more':>5} words ({mask.sum():4d} articles): "
              f"accuracy {(pred[mask] == y[mask]).mean():.4f}")

device: cuda


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]


validation accuracy: 0.9724

--- Calibration ---
model said 0.0-0.1 (3405 articles): average claimed 0.00, really fake 0.02
model said 0.1-0.2 (  12 articles): average claimed 0.13, really fake 0.17
model said 0.2-0.3 (   7 articles): average claimed 0.25, really fake 0.71
model said 0.3-0.4 (  16 articles): average claimed 0.34, really fake 0.50
model said 0.4-0.5 (   6 articles): average claimed 0.45, really fake 0.50
model said 0.5-0.6 (   7 articles): average claimed 0.55, really fake 0.71
model said 0.6-0.7 (   9 articles): average claimed 0.66, really fake 0.78
model said 0.7-0.8 (  12 articles): average claimed 0.74, really fake 0.50
model said 0.8-0.9 (  16 articles): average claimed 0.86, really fake 0.69
model said 0.9-1.0 (2625 articles): average claimed 1.00, really fake 0.97
average gap between claimed and real (ECE): 0.0231   (0 = perfectly honest)

--- Abstention: accuracy on the articles we still answer ---
confidence >= 0.50: answered 100.0% | accuracy 0.9724
confiden

# Does the normalized model stay confident on text it was not trained for


In [ ]:
import re

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

BASE = "/content/drive/MyDrive/fake-news"
MODEL_DIR = f"{BASE}/models/distilroberta_v3_normalized"
device = "cuda" if torch.cuda.is_available() else "cpu"


def normalize(text):
    """Same function used to train the normalized model."""
    text = text.lower()
    text = text.replace("’", "'").replace("‘", "'").replace("“", '"').replace("”", '"')
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


tok = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(device).eval()


def predict_fake(texts, batch_size=64):
    """Return the probability that each text is fake."""
    out = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = tok(texts[i:i + batch_size], truncation=True, max_length=256,
                        padding=True, return_tensors="pt").to(device)
            out.append(torch.softmax(model(**batch).logits, dim=-1)[:, 1].cpu().numpy())
    return np.concatenate(out)


def report(name, p, y):
    pred = (p >= 0.5).astype(int)
    conf = np.maximum(p, 1 - p)
    print(f"\n=== {name} ({len(y)} texts) ===")
    print(f"accuracy: {(pred == y).mean():.4f} | said fake for {pred.mean():.1%} (truly fake: {y.mean():.1%})")
    for lo, hi in [(0.5, 0.9), (0.9, 0.99), (0.99, 1.01)]:
        mask = (conf >= lo) & (conf < hi)
        if mask.sum():
            print(f"  confidence {lo:.2f}-{min(hi, 1):.2f}: {mask.mean():6.1%} of texts, "
                  f"accuracy {(pred[mask] == y[mask]).mean():.4f}")


# 1. LIAR statements (fact-checked claims, never seen in training)
liar = pd.read_csv(f"{BASE}/liar/liar_eval.csv")
p_liar = predict_fake([normalize(s) for s in liar["statement"]])
report("LIAR statements", p_liar, liar["label"].values)

# 2. Validation articles cut to their first N words, to imitate short inputs.
# LIAR statements average about 18 words.
val = pd.read_csv(f"{BASE}/processed_v3/val.csv")
val["title"] = val["title"].fillna("")
val["text"] = val["text"].fillna("")
full = (val["title"] + ". " + val["text"]).tolist()
y_val = val["label"].values

for n in (18, 40, 80):
    cut = [" ".join(t.split()[:n]) for t in full]
    report(f"validation articles cut to the first {n} words",
           predict_fake([normalize(t) for t in cut]), y_val)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]


=== LIAR statements (8072 texts) ===
accuracy: 0.4642 | said fake for 88.4% (truly fake: 43.9%)
  confidence 0.50-0.90:   9.8% of texts, accuracy 0.4784
  confidence 0.90-0.99:  19.1% of texts, accuracy 0.4412
  confidence 0.99-1.00:  71.2% of texts, accuracy 0.4684

=== validation articles cut to the first 18 words (6115 texts) ===
accuracy: 0.8334 | said fake for 58.8% (truly fake: 43.7%)
  confidence 0.50-0.90:   6.2% of texts, accuracy 0.5356
  confidence 0.90-0.99:  10.6% of texts, accuracy 0.5193
  confidence 0.99-1.00:  83.2% of texts, accuracy 0.8955

=== validation articles cut to the first 40 words (6115 texts) ===
accuracy: 0.9395 | said fake for 47.1% (truly fake: 43.7%)
  confidence 0.50-0.90:   2.6% of texts, accuracy 0.5155
  confidence 0.90-0.99:   4.2% of texts, accuracy 0.5512
  confidence 0.99-1.00:  93.2% of texts, accuracy 0.9688

=== validation articles cut to the first 80 words (6115 texts) ===
accuracy: 0.9599 | said fake for 44.2% (truly fake: 43.7%)
  confide